<div align="center">

# Fold a classifier into word vectors

Fold a trained classifier into word vectors, then save a smaller model for your app.

</div>

## When do you need this?

You have a linear text classifier and only need labels or class probabilities in your app.
You want to avoid building a full sentence vector before applying its head.
**Prepare word scores once**, then let fse average those scores when new texts arrive.

## The story in pictures

![A separate path averages word vectors before applying a linear head. A folded path averages prepared word scores and adds the same bias.](media/08-folded-classifier.svg)

A **linear head** turns a sentence vector into one score per class by multiplying it by learned weights and adding a bias.
The bias is a fixed score added after averaging.
**Folding** moves the multiplication into the word table ahead of time, while keeping the bias separate.
For four news topics, each word can then supply four class scores instead of 256 vector values.

## 1. Start with raw sentence means

Use the [shared setup](README.md#run-a-recipe) and the real POTION-base-8M model.
A **raw mean** averages word vectors without scaling the resulting sentence vector to unit length.
This matters because a linear head can fold through an average, but not through that sentence-dependent scaling.
Unlike recipe 02, we train this head on raw means from the start.

In [1]:
from _helpers import Recipe
import fse
import numpy as np

recipe = Recipe()
work = recipe.work

Recipe run: 2026-11-01. Generated files are temporary.
Apple M1 Pro. macOS-27.0.1-arm64-arm-64bit-Mach-O. Python 3.13.16.


`Encoder.from_pretrained(..., normalize=False)` works with every supported model, including classic Hub models under `fse/`.
Our classic Hub models load normalized by default. Load with `normalize=False` before training a head for folding.
A classic model name ending in `-raw` means no SIF weighting, not unnormalized sentence vectors.

In [2]:
encoder = fse.Encoder.from_pretrained(recipe.model_folder(), normalize=False)
assert "normalize@1" not in encoder.steps
print("Values per word:", encoder.output_dim)
print("Sentence steps:", encoder.steps)

Values per word: 256
Sentence steps: ()


/Users/oliverborchers/.herdr/worktrees/Fast_Sentence_Embeddings/dr195-tutorials/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Keep training and test articles separate

We use [AG News](https://huggingface.co/datasets/SetFit/ag_news), which assigns articles to four topics.
The pinned dataset revision comes from the shared helper.
**Use the same fit and test rows as recipe 02**, with seed 7, so a different data selection does not obscure the example.

In [3]:
from sklearn.model_selection import train_test_split

SEED = 7
train_texts, train_labels = recipe.dataset("AG News", "train")
test_texts, test_labels = recipe.dataset("AG News", "test")
train_texts, _, train_labels, _ = train_test_split(
    train_texts, train_labels, train_size=8000, stratify=train_labels, random_state=SEED,
)
test_texts, _, test_labels, _ = train_test_split(
    test_texts, test_labels, train_size=2000, stratify=test_labels, random_state=SEED,
)
fit_texts, _, fit_labels, _ = train_test_split(
    train_texts, train_labels, test_size=0.2, stratify=train_labels, random_state=SEED,
)
print({"fit_articles": len(fit_texts), "test_articles": len(test_texts)})

{'fit_articles': 6400, 'test_articles': 2000}


## 3. Train one head and use the separate path

Train logistic regression in scikit-learn, which learns weights for the four topic scores.
First, encode each test article into 256 numbers, then let the fitted head predict its label and probabilities.
**This is our reference path**, using the same head we will fold below.

In [4]:
from sklearn.linear_model import LogisticRegression

fit_vectors = encoder.encode(fit_texts, workers=1)
head = LogisticRegression(C=1, max_iter=1000, random_state=SEED).fit(fit_vectors, fit_labels)
test_vectors = encoder.encode(test_texts, workers=1)
reference_labels = head.predict(test_vectors)
reference_probabilities = head.predict_proba(test_vectors)
print("Test vector matrix:", test_vectors.shape)
print("Head weights:", head.coef_.shape)

Test vector matrix: (2000, 256)
Head weights: (4, 256)


## 4. Let fse fold the same head

Importing the head does not train anything or change the word vectors used for training.
On its first prediction, **fse prepares the folded word table automatically** for a head on raw means.
It averages the prepared word scores, adds the bias, and applies the probability rule.

In [5]:
classifier = fse.Classifier.from_sklearn(encoder, head)
assert classifier.normalized is False
folded_labels = classifier.predict(test_texts, workers=1)
folded_probabilities = classifier.predict_proba(test_texts, workers=1)
np.testing.assert_allclose(folded_probabilities, reference_probabilities, atol=3e-6, rtol=3e-6)
print("Scores per folded word:", classifier.n_outputs)
print("Largest probability difference:", float(np.max(np.abs(
    folded_probabilities - reference_probabilities,
))))
print("Label disagreements with the separate path:", int(np.count_nonzero(
    folded_labels != reference_labels,
)))

Scores per folded word: 4
Largest probability difference: 3.069639205932617e-06
Label disagreements with the separate path: 0


### Why can this work?

For a linear head, multiplying an average gives the same mathematical result as averaging the multiplied rows.
The [fse model format](../docs/architecture/formats.md) stores that prepared table and adds the bias after averaging.
**Float32 rounding can differ between the two paths**, so the code checks probabilities rather than assuming bit identity with scikit-learn.
Labels near a decision boundary can differ, and the printed disagreement count makes that visible.

## 5. Save both deployment formats

A **complete model** stores the original encoder and the trained head together.
A **standalone model** stores the folded word scores, bias, tokenizer, and labels without the original vector table.
Both fse formats already use the folded prediction path for this raw-mean head.
**Standalone saves space**, rather than enabling an extra speed mode.

In [6]:
complete_path = work / "news-complete.safetensors"
standalone_path = work / "news-standalone.safetensors"
classifier.save(complete_path)
classifier.save(standalone_path, standalone=True)
print("Saved complete and standalone classifiers.")

Saved complete and standalone classifiers.


## 6. Load the files and check their predictions

Use `Classifier.load` for either file, rather than `Encoder.load`.
A standalone classifier cannot return the original sentence vectors because it no longer contains that table.
**Saving and loading preserve fse's prediction bits**, which we check for both formats.

In [7]:
complete = fse.Classifier.load(complete_path)
standalone = fse.Classifier.load(standalone_path)
assert complete.standalone is False and standalone.standalone is True
for loaded in (complete, standalone):
    np.testing.assert_array_equal(loaded.predict(test_texts, workers=1), folded_labels)
    actual = loaded.predict_proba(test_texts, workers=1)
    np.testing.assert_array_equal(actual.view(np.uint32), folded_probabilities.view(np.uint32))
print("Both loaded classifiers preserve every label and probability bit.")

Both loaded classifiers preserve every label and probability bit.


## Result: quality before and after folding

Accuracy is the fraction of articles assigned the correct topic.
Macro-F1 averages a score for each topic equally, so a large class cannot dominate the result.
**Compare the same fitted raw-mean head across these paths**, rather than comparing different training methods.

In [8]:
from sklearn.metrics import accuracy_score, f1_score
from IPython.display import Markdown, display

paths = {
    "Separate encoder + scikit-learn head": reference_labels,
    "fse, folded automatically": folded_labels,
    "Reloaded complete fse model": complete.predict(test_texts, workers=1),
    "Reloaded standalone fse model": standalone.predict(test_texts, workers=1),
}
display(Markdown("| Prediction path | Accuracy | Macro-F1 |\n|---|---:|---:|\n" + "\n".join(
    f"| {name} | {accuracy_score(test_labels, labels):.4f} | "
    f"{f1_score(test_labels, labels, average='macro'):.4f} |" for name, labels in paths.items()
)))

| Prediction path | Accuracy | Macro-F1 |
|---|---:|---:|
| Separate encoder + scikit-learn head | 0.8565 | 0.8564 |
| fse, folded automatically | 0.8565 | 0.8564 |
| Reloaded complete fse model | 0.8565 | 0.8564 |
| Reloaded standalone fse model | 0.8565 | 0.8564 |

The raw-mean head is a different trained model from recipe 02's head on unit-length vectors.
Changing normalization can change quality, so evaluate that choice separately from folding.
The label and probability checks above isolate the effect of folding this one head.

## Result: saved model size

Measure the actual files, including their tokenizer and labels.
A smaller word table does not imply the same reduction for the whole file, and **file size is not peak RAM use**.

In [9]:
display(Markdown("| Saved format | Bytes | MiB | Original sentence vectors available? |\n"
                 "|---|---:|---:|---|\n" + "\n".join(
    f"| {name} | {path.stat().st_size:,} | {path.stat().st_size / 2**20:.3f} | {vectors} |"
    for name, path, vectors in (
        ("Complete", complete_path, "yes"),
        ("Standalone", standalone_path, "no"),
    )
)))

| Saved format | Bytes | MiB | Original sentence vectors available? |
|---|---:|---:|---|
| Complete | 30,668,970 | 29.248 | yes |
| Standalone | 900,522 | 0.859 | no |

## What about speed?

The folded path reads four prepared scores per word here, while the separate path first averages 256 vector values.
That reduces the width of the word table used for prediction, but it does not establish a measured speedup.
**No runtime comparison is reported yet** because the matching committed raw measurements are not available.
The [performance guide](../PERFORMANCE.md) explains the existing measurements and their limits.

## Use the standalone file in your app

Your application can load the finished classifier and predict without importing scikit-learn.
The labels below use topic names so you can inspect real predictions beside their articles.

In [10]:
deployed = fse.Classifier.load(standalone_path)
topic_names = ["World", "Sports", "Business", "Science and technology"]
for text, label in zip(test_texts[:3], deployed.predict(test_texts[:3], workers=1), strict=True):
    print(topic_names[int(label)] + ": " + text[:160])

Business: Heineken Profit Dips But Repeats Outlook  AMSTERDAM (Reuters) - Dutch brewer Heineken posted a 4.5  percent fall in core profit for the first half on Wednesday,
World: The radical Islamic cleric was deported from Germany late Tuesday &lt;b&gt;...&lt;/b&gt; Turkish officials were doing what was necessary in regard to the return
World: Hamm should support Yang Paul Hamm needs a new marketing strategy. Either that, or he needs a clue. One harmless gesture separates him from lionization in Ameri


## What next?

- [Compare classifier training methods](02-linear-heads.ipynb).
- [Read the linear-head format and numerical contract](../docs/architecture/formats.md).
- [Train a scikit-learn linear model](https://scikit-learn.org/stable/modules/linear_model.html).

Back to the [recipe index](README.md) or the [main README](../README.md).

## Clean up

The helper removes both saved models and downloaded data from the temporary workspace.
Use permanent paths in your app and copy the file you want to keep before running this cell.

In [11]:
recipe.close()